# Heart (MRI) Baseline - UniverSeg In-Context Evaluation (30 Test Slices)

In [ ]:
import sys, os, torch, re
import numpy as np
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../external/universeg'))
from universeg import universeg


In [ ]:
images = np.load('../data/processed/heart_images.npy')
masks = np.load('../data/processed/heart_masks.npy')
print(f'Loaded heart images shape:', images.shape)
print(f'Loaded heart masks shape: ', masks.shape)


In [ ]:
np.random.seed(42)
torch.manual_seed(42)

total_n = len(images)
all_indices = np.arange(total_n)
support_indices = np.random.choice(all_indices, size=2, replace=False)
remaining_indices = np.setdiff1d(all_indices, support_indices)
test_indices = np.random.choice(remaining_indices, size=30, replace=False)

print(f'Support Set Slices (2): {support_indices}')
print(f'Test Set Slices (30):    {test_indices}')


In [ ]:
def preprocess_slice(img, mask):
    t_img = torch.from_numpy(img.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    t_msk = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    t_img_res = TF.resize(t_img, [128, 128], antialias=True)
    t_msk_res = TF.resize(t_msk, [128, 128], antialias=False)
    return t_img_res.squeeze(0), t_msk_res.squeeze(0)

supp_imgs, supp_msks = [], []
for idx in support_indices:
    si, sm = preprocess_slice(images[idx], masks[idx])
    supp_imgs.append(si)
    supp_msks.append(sm)

support_images_t = torch.stack(supp_imgs, dim=0).unsqueeze(0)
support_masks_t = torch.stack(supp_msks, dim=0).unsqueeze(0)

model = universeg(pretrained=True).eval()

def compute_dice(pred, target, eps=1e-6):
    intersection = (pred * target).sum()
    total = pred.sum() + target.sum()
    return float((2.0 * intersection + eps) / (total + eps))

results = []
for i, idx in enumerate(test_indices):
    ti, tm = preprocess_slice(images[idx], masks[idx])
    target_img_t = ti.unsqueeze(0)
    with torch.no_grad():
        logits = model(target_img_t, support_images_t, support_masks_t)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
    score = compute_dice(preds[0, 0].cpu().numpy(), tm[0].cpu().numpy())
    results.append({
        'test_idx': idx,
        'ct_img': images[idx],
        'gt_mask': tm[0].cpu().numpy(),
        'pred_mask': preds[0, 0].cpu().numpy(),
        'dice': score
    })

scores = [r['dice'] for r in results]
mean_dice = np.mean(scores)
std_dice = np.std(scores)
print(f'Mean Dice Score (30 test slices): {mean_dice:.4f} +/- {std_dice:.4f}')


In [ ]:
# Save score logs
os.makedirs('../logs', exist_ok=True)
log_path = '../logs/heart_baseline_scores.txt'
with open(log_path, 'w') as f:
    f.write('UniverSeg Baseline Results - Heart (MRI) Baseline
')
    f.write('='*45 + '
')
    f.write(f'Support Slice Indices: {list(support_indices)}
')
    f.write(f'Test Slice Indices (30): {list(test_indices)}

')
    for idx_i, (idx, d) in enumerate(zip(test_indices, scores)):
        f.write(f'Test Slice {idx_i+1:02d} (Index {idx:5d}): Dice = {d:.4f}
')
    f.write(f'
Mean Dice Score: {mean_dice:.4f}
')
    f.write(f'Std Dice Score:  {std_dice:.4f}
')
print(f'Saved scores log to {log_path}')

# Display 3 visual comparisons
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
plt.suptitle(f'Heart (MRI) Baseline Baseline Results (Mean Dice: {mean_dice:.4f} +/- {std_dice:.4f})', fontsize=14, y=0.995)

for idx_i in range(3):
    res = results[idx_i]
    t_idx = res['test_idx']
    d_score = res['dice']
    axes[idx_i, 0].imshow(res['ct_img'], cmap='gray')
    axes[idx_i, 0].set_title(f'Test Slice {idx_i+1} (idx {t_idx})')
    axes[idx_i, 0].axis('off')
    
    axes[idx_i, 1].imshow(res['gt_mask'], cmap='Blues', vmin=0, vmax=1)
    axes[idx_i, 1].set_title('Ground Truth Mask')
    axes[idx_i, 1].axis('off')
    
    axes[idx_i, 2].imshow(res['pred_mask'], cmap='Oranges', vmin=0, vmax=1)
    axes[idx_i, 2].set_title(f'UniverSeg Pred (Dice: {d_score:.4f})')
    axes[idx_i, 2].axis('off')

plt.tight_layout()
os.makedirs('../reports', exist_ok=True)
plt.savefig(f'../reports/heart_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
